# Visualization and Export Utilities
<!-- structured-notebook -->
## Notebook Summary
Purpose: collection of small visualization and data-export utilities used to produce figures for the poster, paper, and internal review slides.

Main steps:
- Draw UMAP scatter plots and topic-distribution visuals.
- Produce poster-specific figures with different styling constraints.
- Plot temporal distributions by year.
- Visualize high-topic-count models (the 300-topic pre-narrowing run).
- Export topic keywords for external use.
- Check post-length distributions as a corpus quality probe.


# Main Topic Visualization

In [ ]:
from src.shared_reddit_telegram.config import (
    PUBMED_MODELS,
    REDDIT_MODELS,
)
import webbrowser

from bertopic import BERTopic

# Load the trained BERTopic model
model_path = "round_10/bertopic_no_embed"
topic_model = BERTopic.load(model_path)

# Visualize topics (overview)
fig = topic_model.visualize_topics()
fig2 = topic_model.visualize_barchart(top_n_topics=30, n_words=10)

# Save visualization to HTML
fig.write_html("round_10/topic_overview.html")
fig2.write_html("round_10/topic_barchart.html")

webbrowser.open(f"file://{REDDIT_MODELS / 'round_10/topic_overview.html'}")
webbrowser.open(f"file://{REDDIT_MODELS / 'round_10/topic_barchart.html'}")



# Poster-Specific Visuals

In [ ]:
# -*- coding: utf-8 -*-
"""
Reddit BERTopic poster visuals: 1–8
Now selectable via VISUALS toggles.
Outputs: round_5/figs/*.png and round_5/exports/*.csv
"""

import os
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# ========= Which visuals to run? =========
# Set any to False to skip generating that visual/export.
VISUALS = {
    1: True,  # UMAP topic landscape
    2: False,  # Top topics by volume & engagement
    3: False,  # Topic trends over time
    4: False,  # Subreddit × topic heatmap
    5: False,  # Sentiment by topic (requires vaderSentiment)
    6: False,  # Topic co-occurrence network
    7: False,  # Vocabulary cards
    8: True,  # Lifecycle metrics (emergence, peak, half-life)
}

# ========= Paths =========
ROUND = 10
BASE = PUBMED_MODELS / "10000_docs_allenai_umap_hdbscan/BERTopic_model"
FIGS = f"{BASE}/figs"
EXPO = f"{BASE}/exports"
os.makedirs(FIGS, exist_ok=True)
os.makedirs(EXPO, exist_ok=True)

# ========= Load core data you already produced =========
# meta has K rows, aligned 1:1 with topics_K/ts_K
meta = pd.read_parquet(f"{BASE}/kept_metadata.parquet")
topics = meta["topic_id"].to_numpy(np.int32)
ts = pd.to_datetime(meta["pub_date"], utc=True)
subreddits = meta["journal"].astype(str).fillna("unknown")

# optional 2D coords (UMAP of embeddings), if saved:
coords_2d_path = f"{BASE}/umap2d.npy"
coords_2d = np.load(coords_2d_path) if os.path.exists(coords_2d_path) else None

# topic info (words per topic)
topics_info = pd.read_csv(f"{BASE}/topics.csv")   # columns: Topic, Count, Name, etc.
valid_topics = set(topics_info["Topic"].unique())

# helper: short labels for topics
def make_short_labels(topics_info, max_len=20):
    labels = {}
    for _, row in topics_info.iterrows():
        t = int(row["Topic"])
        if t == -1:
            labels[t] = "OUTLIER"
            continue
        # try 'Name' column if present and non-empty; else derive from 'Representation' or keywords
        name = str(row.get("Name", "")).strip()
        if not name or name == "nan":
            # fallback: use "Top_n_words" if exists or try parse from topics_info
            # Many BERTopic versions store top words in another dataframe,
            # here we just use row's string if available
            name = str(row.get("Representation", "")).strip()
        if not name or name == "nan":
            name = f"Topic {t}"
        # keep it short
        labels[t] = (name[:max_len] + "…") if len(name) > max_len else name
    return labels

topic_label = make_short_labels(topics_info)

# helper: safe color per topic (consistent across figs)
# keep simple: cycle through matplotlib prop cycle but anchor colors to topic id
def topic_color(tid):
    # deterministic color selection from Matplotlib default cycle
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    if tid == -1:
        return "lightgray"
    return colors[(tid * 9973) % len(colors)]

# ======== 1) Topic landscape (UMAP 2D) ========
def fig_topic_landscape(coords_2d, topics, topic_label, max_points=80000, s=4):
    if coords_2d is None:
        print("[1] Skipped UMAP scatter: coords_2d not found.")
        return
    n = coords_2d.shape[0]
    sel = np.arange(n)
    if n > max_points:
        rng = np.random.default_rng(42)
        sel = rng.choice(n, size=max_points, replace=False)
    x = coords_2d[sel, 0]
    y = coords_2d[sel, 1]
    t = topics[sel]

    plt.figure(figsize=(9, 7), dpi=150)
    # plot by topic in batches to avoid massive legend
    for tid in sorted(set(t)):
        mask = (t == tid)
        plt.scatter(x[mask], y[mask], s=s, alpha=0.5, label=topic_label.get(int(tid), f"Topic {tid}"),
                    c=topic_color(int(tid)))
    plt.title("Topic landscape (UMAP 2D) – Reddit")
    plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
    # Make a compact legend by sampling a few topics
    handles, labels = plt.gca().get_legend_handles_labels()
    if len(handles) > 20:
        # show top 15 by overall count in selection (plus OUTLIER if present)
        counts = Counter(t.tolist())
        top_ids = [tid for tid,_ in counts.most_common(14)]
        if -1 in counts and -1 not in top_ids: top_ids = top_ids[:-1] + [-1]
        keep = []
        for tid in top_ids:
            name = topic_label.get(int(tid), f"Topic {tid}")
            for h, lab in zip(handles, labels):
                if lab == name:
                    keep.append((h, lab)); break
        handles, labels = zip(*keep)
    plt.legend(handles, labels, loc="best", fontsize=8, framealpha=0.8, markerscale=3)
    plt.tight_layout()
    plt.savefig(f"{FIGS}/01_topic_landscape_umap.png")
    plt.close()
    print("[1] Saved:", f"{FIGS}/01_topic_landscape_umap.png")

if VISUALS.get(1, False):
    fig_topic_landscape(coords_2d, topics, topic_label)

# ======== Helpers for aggregations ========
# Time binning (monthly)
meta["_month"] = ts.dt.to_period("M").astype(str)

# Engagement metric (tweak weights as you like)
def engagement_score(df):
    # Composite: log1p(score) and log1p(num_comments)
    s = np.log1p(df["score"].fillna(0).astype(float))
    c = np.log1p(df["num_comments"].fillna(0).astype(float))
    return 0.5 * s + 0.5 * c

#Uncomment for Reddit
#meta["_engagement"] = engagement_score(meta)

# ======== 2) Top topics by volume & engagement ========
def fig_top_topics_volume_engagement(meta, topic_label, top_k=15):
    grp = meta.groupby("topic_hard").agg(
        volume=("topic_hard", "size"),
        med_eng=("_engagement", "median"),
        mean_eng=("_engagement", "mean")
    ).reset_index().rename(columns={"topic_hard": "topic"})
    grp["label"] = grp["topic"].map(lambda t: topic_label.get(int(t), f"Topic {t}"))

    grp.to_csv(f"{EXPO}/02_topic_volume_engagement.csv", index=False)

    # Volume bar
    top_vol = grp.sort_values("volume", ascending=False).head(top_k)
    plt.figure(figsize=(9, 6), dpi=150)
    plt.barh(top_vol["label"][::-1], top_vol["volume"][::-1],
             color=[topic_color(int(t)) for t in top_vol["topic"][::-1]])
    plt.title("Top topics by volume")
    plt.xlabel("Posts"); plt.ylabel("")
    plt.tight_layout(); plt.savefig(f"{FIGS}/02a_top_topics_volume.png"); plt.close()

    # Engagement bar (median)
    top_eng = grp.sort_values("med_eng", ascending=False).head(top_k)
    plt.figure(figsize=(9, 6), dpi=150)
    plt.barh(top_eng["label"][::-1], top_eng["med_eng"][::-1],
             color=[topic_color(int(t)) for t in top_eng["topic"][::-1]])
    plt.title("Top topics by engagement (median composite)")
    plt.xlabel("Median engagement"); plt.ylabel("")
    plt.tight_layout(); plt.savefig(f"{FIGS}/02b_top_topics_engagement.png"); plt.close()

    print("[2] Saved:",
          f"{FIGS}/02a_top_topics_volume.png",
          f"{FIGS}/02b_top_topics_engagement.png")

if VISUALS.get(2, False):
    fig_top_topics_volume_engagement(meta, topic_label)

# ======== 3) Topic trends over time (small multiples for top N) ========
def fig_topic_trends(meta, topic_label, top_n=12, smooth=3):
    ts_topic = (meta
                .groupby(["_month", "topic_hard"])
                .size()
                .reset_index(name="count"))
    # normalize within month if you prefer shares:
    # month_tot = ts_topic.groupby("_month")["count"].transform("sum")
    # ts_topic["share"] = ts_topic["count"] / month_tot

    # pick top topics by total volume
    top_topics = (ts_topic.groupby("topic_hard")["count"].sum()
                  .sort_values(ascending=False).head(top_n).index.tolist())
    plot_df = ts_topic[ts_topic["topic_hard"].isin(top_topics)].copy()
    plot_df["_month_dt"] = pd.to_datetime(plot_df["_month"])
    plot_df = plot_df.sort_values("_month_dt")

    # optional smoothing
    def moving_avg(x, w):
        return np.convolve(x, np.ones(w)/w, mode="same") if w > 1 else x

    plt.figure(figsize=(11, 8), dpi=150)
    for tid in top_topics:
        d = plot_df[plot_df["topic_hard"] == tid].set_index("_month_dt")["count"]
        # fill missing months with 0 for continuity
        full_idx = pd.period_range(plot_df["_month"].min(), plot_df["_month"].max(), freq="M").to_timestamp()
        d = d.reindex(full_idx, fill_value=0)
        y = moving_avg(d.values, smooth)
        plt.plot(full_idx, y, label=topic_label.get(int(tid), f"Topic {tid}"),
                 linewidth=1.8, color=topic_color(int(tid)))
    plt.title("Topic trends over time (monthly, smoothed)")
    plt.xlabel(""); plt.ylabel("Posts / month")
    plt.legend(fontsize=8, ncol=2, framealpha=0.8)
    plt.tight_layout(); plt.savefig(f"{FIGS}/03_topic_trends_smallmultiples.png"); plt.close()

    plot_df.to_csv(f"{EXPO}/03_topic_trends_monthly_counts.csv", index=False)
    print("[3] Saved:", f"{FIGS}/03_topic_trends_smallmultiples.png")

if VISUALS.get(3, False):
    fig_topic_trends(meta, topic_label)

# ======== 4) Subreddit × topic heatmap ========
def fig_subreddit_topic_heatmap(meta, topic_label, max_subs=20, top_topics=20, zscore_by="row"):
    # pick top subreddits by volume
    top_subs = meta["subreddit"].value_counts().head(max_subs).index.tolist()
    df = meta[meta["subreddit"].isin(top_subs)]
    # topic prevalence in each subreddit
    cross = (df.groupby(["subreddit", "topic_hard"])
             .size().reset_index(name="count"))
    # choose columns (topics) to show: top by global count
    top_t = (cross.groupby("topic_hard")["count"].sum()
             .sort_values(ascending=False).head(top_topics).index.tolist())
    cross = cross[cross["topic_hard"].isin(top_t)]
    # pivot to matrix
    M = cross.pivot(index="subreddit", columns="topic_hard", values="count").fillna(0.0)
    # convert to share per subreddit (row-normalize)
    row_sum = M.sum(axis=1).replace(0, 1)
    share = (M.T / row_sum).T
    # z-score per subreddit row for contrast (optional)
    if zscore_by == "row":
        mu = share.mean(axis=1)
        sd = share.std(axis=1).replace(0, 1e-9)
        Z = ((share.T - mu) / sd).T
        mat = Z
        title = "Subreddit × topic (row z-score of topic share)"
    else:
        mat = share
        title = "Subreddit × topic (topic share per subreddit)"

    # plot
    plt.figure(figsize=(12, 7), dpi=150)
    im = plt.imshow(mat.values, aspect="auto", interpolation="nearest")
    plt.colorbar(im, fraction=0.02, pad=0.02)
    plt.yticks(range(mat.shape[0]), mat.index)
    col_labels = [topic_label.get(int(t), f"T{t}") for t in mat.columns]
    plt.xticks(range(mat.shape[1]), col_labels, rotation=65, ha="right")
    plt.title(title)
    plt.tight_layout(); plt.savefig(f"{FIGS}/04_subreddit_topic_heatmap.png"); plt.close()

    mat.to_csv(f"{EXPO}/04_subreddit_topic_matrix.csv")
    print("[4] Saved:", f"{FIGS}/04_subreddit_topic_heatmap.png")

if VISUALS.get(4, False):
    fig_subreddit_topic_heatmap(meta, topic_label)

# ======== 5) Sentiment by topic (stacked bars) ========
# We'll use VADER on title + selftext for speed/robustness.
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    HAVE_VADER = True
except Exception:
    HAVE_VADER = False
    print("[5] vaderSentiment not installed. Run: pip install vaderSentiment")

def compute_vader_sentiment(meta):
    if not HAVE_VADER:
        return None
    ana = SentimentIntensityAnalyzer()
    texts = (meta["title"].fillna("").astype(str) + " " + meta["selftext"].fillna("").astype(str)).tolist()
    comp = []
    for s in texts:
        ss = ana.polarity_scores(s)
        comp.append(ss["compound"])
    meta["_sent_comp"] = np.array(comp, dtype=np.float32)
    # label bins
    lab = np.where(meta["_sent_comp"] >= 0.05, "pos",
           np.where(meta["_sent_comp"] <= -0.05, "neg", "neu"))
    meta["_sent_label"] = lab
    return meta

def fig_sentiment_by_topic(meta, topic_label, min_docs=50, order_by="pos"):
    if "_sent_label" not in meta.columns:
        print("[5] Skipped sentiment plot: no sentiment computed.")
        return
    # aggregate
    g = (meta.groupby(["topic_hard", "_sent_label"])
              .size()
              .unstack("_sent_label")
              .fillna(0.0))
    g["total"] = g.sum(axis=1)
    g = g[g["total"] >= min_docs]
    for col in ("pos", "neu", "neg"):
        if col not in g.columns: g[col] = 0.0
    g["pos_share"] = g["pos"] / g["total"]
    g["neg_share"] = g["neg"] / g["total"]
    g["neu_share"] = g["neu"] / g["total"]

    # order topics
    if order_by == "pos":
        g = g.sort_values("pos_share", ascending=False)
    elif order_by == "neg":
        g = g.sort_values("neg_share", ascending=False)

    labels = [topic_label.get(int(t), f"T{t}") for t in g.index]
    plt.figure(figsize=(10, 7), dpi=150)
    bottom = np.zeros(len(g))
    for col in ["neg_share", "neu_share", "pos_share"]:
        plt.barh(labels, g[col].values, left=bottom, label=col.replace("_share","").upper())
        bottom += g[col].values
    plt.legend(loc="lower right")
    plt.title("Sentiment by topic (VADER)")
    plt.xlabel("Share"); plt.ylabel("")
    plt.tight_layout(); plt.savefig(f"{FIGS}/05_sentiment_by_topic.png"); plt.close()

    g.to_csv(f"{EXPO}/05_sentiment_by_topic.csv")
    print("[5] Saved:", f"{FIGS}/05_sentiment_by_topic.png")

if VISUALS.get(5, False):
    if HAVE_VADER:
        meta = compute_vader_sentiment(meta)
        fig_sentiment_by_topic(meta, topic_label)
    else:
        print("[5] Sentiment figure skipped (install vaderSentiment to enable).")

# ======== 6) Topic co-occurrence network ========
# Define co-occurrence unit = same subreddit-month (robust for submissions)
def compute_topic_cooccurrence(meta, unit_cols=("subreddit", "_month"), min_edge=10, top_k_per_node=5):
    df = meta.copy()
    df["_unit"] = df.groupby(list(unit_cols)).ngroup()
    # collect topics per unit
    # Use set of topics per unit to avoid overcounting duplicates within unit
    unit_topics = df.groupby("_unit")["topic_hard"].apply(lambda s: sorted(set(s.tolist())))
    # accumulate pair counts
    pair_count = Counter()
    for topics_in_unit in unit_topics:
        if len(topics_in_unit) < 2:
            continue
        # all unordered pairs
        for i in range(len(topics_in_unit)):
            for j in range(i+1, len(topics_in_unit)):
                a, b = topics_in_unit[i], topics_in_unit[j]
                pair_count[(a, b)] += 1

    # build dataframe
    rows = []
    for (a, b), c in pair_count.items():
        if c >= min_edge:
            rows.append((a, b, c))
    co = pd.DataFrame(rows, columns=["t_i", "t_j", "weight"])
    if co.empty:
        return co

    # sparsify: keep top_k edges per node by weight
    keep = set()
    for col in ["t_i", "t_j"]:
        tmp = co.groupby(col).apply(lambda g: g.nlargest(top_k_per_node, "weight").index)
        for idxs in tmp:
            keep.update(idxs.tolist())
    co = co.loc[sorted(keep)]
    return co.sort_values("weight", ascending=False)

def fig_cooccurrence_network(meta, topic_label, min_edge=10, top_k_per_node=5):
    co = compute_topic_cooccurrence(meta, ("subreddit", "_month"),
                                    min_edge=min_edge, top_k_per_node=top_k_per_node)
    co.to_csv(f"{EXPO}/06_topic_cooccurrence_edges.csv", index=False)

    if co.empty:
        print("[6] Co-occurrence: no edges meeting threshold.")
        return

    # Simple hairball-less layout: place nodes on a circle; edge thickness ~ weight
    nodes = sorted(set(co["t_i"]).union(set(co["t_j"])))
    n = len(nodes)
    angles = np.linspace(0, 2*math.pi, n, endpoint=False)
    pos = {tid: (math.cos(a), math.sin(a)) for tid, a in zip(nodes, angles)}
    deg = Counter()
    for _, row in co.iterrows():
        deg[row["t_i"]] += row["weight"]
        deg[row["t_j"]] += row["weight"]

    plt.figure(figsize=(9, 9), dpi=150)
    # edges
    max_w = co["weight"].max()
    for _, row in co.iterrows():
        a, b, w = int(row["t_i"]), int(row["t_j"]), row["weight"]
        x1, y1 = pos[a]; x2, y2 = pos[b]
        plt.plot([x1, x2], [y1, y2], linewidth=0.5 + 3.0*(w/max_w), alpha=0.2, color="gray")

    # nodes
    # node size by degree (sum of weights), color by topic
    sizes = []
    X, Y, C, L = [], [], [], []
    for tid in nodes:
        X.append(pos[tid][0]); Y.append(pos[tid][1])
        sizes.append(100 + 600 * (deg[tid] / max(1.0, max(deg.values()))))
        C.append(topic_color(int(tid)))
        L.append(topic_label.get(int(tid), f"T{tid}"))

    plt.scatter(X, Y, s=sizes, c=C, edgecolors="white", linewidths=0.8)
    for (x, y, lab) in zip(X, Y, L):
        plt.text(x, y, lab, fontsize=8, ha="center", va="center")

    plt.axis("off"); plt.title("Topic co-occurrence network (subreddit × month)")
    plt.tight_layout(); plt.savefig(f"{FIGS}/06_topic_cooccurrence_network.png"); plt.close()
    print("[6] Saved:", f"{FIGS}/06_topic_cooccurrence_network.png")

if VISUALS.get(6, False):
    fig_cooccurrence_network(meta, topic_label, min_edge=10, top_k_per_node=5)

# ======== 7) Vocabulary cards (top words + exemplar titles) ========
# Uses topics.csv top words if available; exemplar = highest-score post per topic by engagement
def build_vocab_cards(meta, topics_info, topic_label, top_k=6, top_words=10):
    # pick topics to feature: top by volume
    by_vol = meta["topic_hard"].value_counts().head(top_k).index.tolist()
    # extract top words: assume topics.csv has columns 'Topic' and 'Representation' or similar
    # If your topics.csv stores words differently, adjust parse below.
    words_map = {}
    for _, row in topics_info.iterrows():
        t = int(row["Topic"])
        rep = str(row.get("Representation", "")).strip()
        # Fallback: try a 'Top_n_words' column name if you used a different version
        if not rep or rep == "nan":
            rep = str(row.get("Top_n_words", "")).strip()
        if rep and rep != "nan":
            # try to parse comma/space separated items
            # keep first top_words tokens
            toks = [w.strip() for w in rep.replace("[","").replace("]","").replace("'","").split(",")]
            toks = [w for w in toks if w][:top_words]
            words_map[t] = toks
        else:
            words_map[t] = []
    # exemplars: pick the single post with highest engagement per topic
    exemplars = {}
    for t in by_vol:
        d = meta[meta["topic_hard"] == t].copy()
        if d.empty:
            exemplars[t] = ("", "")
            continue
        d = d.sort_values("_engagement", ascending=False)
        row = d.iloc[0]
        exemplars[t] = (str(row.get("title", ""))[:120], str(row.get("permalink","")))

    # save a compact CSV for poster copy/paste
    rows = []
    for t in by_vol:
        rows.append({
            "topic": t,
            "label": topic_label.get(int(t), f"T{t}"),
            "top_words": ", ".join(words_map.get(t, [])),
            "exemplar_title": exemplars[t][0],
            "exemplar_permalink": exemplars[t][1]
        })
    cards = pd.DataFrame(rows)
    cards.to_csv(f"{EXPO}/07_vocab_cards.csv", index=False)
    return by_vol, words_map, exemplars

def fig_vocab_cards(meta, topics_info, topic_label, top_k=6, top_words=10):
    by_vol, words_map, exemplars = build_vocab_cards(meta, topics_info, topic_label, top_k, top_words)
    # Render a simple “cards” grid as a figure
    cols = 3
    rows = math.ceil(len(by_vol)/cols)
    plt.figure(figsize=(14, 4.5*rows), dpi=150)
    for i, t in enumerate(by_vol):
        ax = plt.subplot(rows, cols, i+1)
        ax.axis("off")
        ttl = topic_label.get(int(t), f"T{t}")
        ax.text(0.0, 1.0, ttl, fontsize=12, weight="bold", va="top", ha="left", transform=ax.transAxes)
        ax.text(0.0, 0.8, "Keywords: " + ", ".join(words_map.get(t, [])), fontsize=10, va="top", ha="left", transform=ax.transAxes)
        ex = exemplars[t][0]
        if ex:
            ax.text(0.0, 0.6, "Exemplar: " + ex, fontsize=10, va="top", ha="left", transform=ax.transAxes,
                    wrap=True)
    plt.suptitle("Vocabulary cards (top topics)", y=0.98)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    out = f"{FIGS}/07_vocab_cards.png"
    plt.savefig(out); plt.close()
    print("[7] Saved:", out)

if VISUALS.get(7, False):
    fig_vocab_cards(meta, topics_info, topic_label, top_k=6, top_words=10)

# ======== 8) Lifecycle metrics: emergence, peak, half-life ========
def topic_lifecycle(meta, min_threshold=10):
    df = (meta.groupby(["_month", "topic_id"])
                .size().reset_index(name="count"))
    df["_month_dt"] = pd.to_datetime(df["_month"])
    liferows = []
    for tid, g in df.groupby("topic_id"):
        g = g.sort_values("_month_dt").set_index("_month_dt")
        # make continuous monthly index
        full_idx = pd.period_range(df["_month"].min(), df["_month"].max(), freq="M").to_timestamp()
        series = g["count"].reindex(full_idx, fill_value=0).astype(int)
        total = series.sum()
        if total < 1:
            continue
        # emergence: first month above threshold (or first nonzero if you prefer)
        thr = max(min_threshold, int(0.01 * total))  # adaptive threshold
        try:
            emergence = series.index[series >= thr][0]
        except IndexError:
            emergence = series.index[series > 0][0] if (series > 0).any() else pd.NaT

        # peak month
        peak_idx = series.idxmax()
        peak_val = series.max()

        # half-life: months after peak until series <= 0.5*peak (first crossing)
        hl = np.nan
        if peak_val > 0:
            target = 0.5 * peak_val
            after = series.loc[peak_idx:]
            # first time falls to or below target (excluding the peak month)
            if len(after) > 1:
                crossing = after.index[(after <= target) & (after.index > peak_idx)]
                if len(crossing) > 0:
                    hl = (crossing[0].to_period("M") - peak_idx.to_period("M")).n
        liferows.append({
            "topic": int(tid),
            "emergence_month": emergence,
            "peak_month": peak_idx,
            "peak_value": int(peak_val),
            "half_life_months": float(hl) if not (hl is np.nan) else np.nan,
            "total_volume": int(total)
        })
    life = pd.DataFrame(liferows).sort_values("total_volume", ascending=False)
    life.to_csv(f"{EXPO}/08_topic_lifecycle.csv", index=False)
    return life

def fig_lifecycle_summary(life, topic_label, top_k=20):
    d = life.head(top_k).copy()
    # Dumbbell: emergence -> peak (x positions), one row per topic
    plt.figure(figsize=(10, 7), dpi=150)
    y = np.arange(len(d))
    x1 = pd.to_datetime(d["emergence_month"])
    x2 = pd.to_datetime(d["peak_month"])
    # convert to matplotlib dates
    import matplotlib.dates as mdates
    x1_m = mdates.date2num(x1)
    x2_m = mdates.date2num(x2)
    for i in range(len(d)):
        plt.plot([x1_m[i], x2_m[i]], [y[i], y[i]], linewidth=2, color="gray", alpha=0.6)
        plt.scatter([x1_m[i], x2_m[i]], [y[i], y[i]],
                    s=[30, 60], c=[topic_color(int(d.iloc[i]["topic"]))]*2, edgecolors="white", linewidths=0.8)
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.yticks(y, [topic_label.get(int(t), f"T{t}") for t in d["topic"]])
    plt.title("Topic lifecycle: emergence → peak (top by volume)")
    plt.xlabel(""); plt.ylabel("")
    plt.tight_layout(); plt.savefig(f"{FIGS}/08a_topic_lifecycle_dumbbell.png"); plt.close()

    # Half-life bar (optional)
    plt.figure(figsize=(8, 6), dpi=150)
    d2 = d[~d["half_life_months"].isna()].sort_values("half_life_months", ascending=True)
    plt.barh([topic_label.get(int(t), f"T{t}") for t in d2["topic"]], d2["half_life_months"],
             color=[topic_color(int(t)) for t in d2["topic"]])
    plt.title("Topic half-life after peak (months)")
    plt.xlabel("Months"); plt.ylabel("")
    plt.tight_layout(); plt.savefig(f"{FIGS}/08b_topic_half_life.png"); plt.close()

    print("[8] Saved:",
          f"{FIGS}/08a_topic_lifecycle_dumbbell.png",
          f"{FIGS}/08b_topic_half_life.png")

if VISUALS.get(8, False):
    life = topic_lifecycle(meta, min_threshold=10)
    fig_lifecycle_summary(life, topic_label)

print("Done.")

# Yearly Distribution Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("round_6/kept_metadata.parquet")

# 1) Pick a timestamp column
for col in ("ts_seconds", "ts_utc", "created_utc"):
    if col in df.columns:
        ts_col = col
        print(col)
        break
else:
    raise ValueError("No timestamp column found (looked for 'timestamp', 'date', 'created_utc').")

# 2) Parse timestamps (assumes epoch seconds if numeric)
if pd.api.types.is_numeric_dtype(df[ts_col]):
    dt = pd.to_datetime(df[ts_col], unit="s", utc=True, errors="coerce")
else:
    dt = pd.to_datetime(df[ts_col], utc=True, errors="coerce")

# Optional: count by local calendar year instead of UTC (uncomment if desired)
# dt = dt.dt.tz_convert("Europe/Budapest")

# 3) Drop missing timestamps
dt = dt.dropna()

# 4) Count messages per year
year_counts = dt.dt.year.value_counts().sort_index()

# 5) Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(year_counts.index.astype(str), year_counts.values)
ax.set_xlabel("Year")
ax.set_ylabel("Number of messages")
ax.set_title("Messages per year")

# Optional: add value labels
for x, y in zip(year_counts.index.astype(str), year_counts.values):
    ax.text(x, y, f"{int(y):,}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

# Visualization for the 300-Topic Pre-Narrowing Run

In [ ]:
import os
import pickle
import random
import webbrowser
from bertopic import BERTopic

def generate_topic_visualizations(
    model_path: str,
    mapping_path: str,
    output_dir: str = "visualizations",
    sample_size: int = 10000
):
    os.makedirs(output_dir, exist_ok=True)

    print("🔄 Loading BERTopic model...")
    topic_model = BERTopic.load(model_path)

    print("📦 Loading topic-document mappings...")
    with open(mapping_path, "rb") as f:
        topic_doc_map = pickle.load(f)

    print("📉 Sampling documents from topic-doc mapping...")
    all_docs = []
    for docs in topic_doc_map.values():
        all_docs.extend(docs)

    if sample_size and len(all_docs) > sample_size:
        all_docs = random.sample(all_docs, sample_size)

    print(f"📄 {len(all_docs)} documents sampled for visualization.")

    # --- Visualizations ---
    visuals = []

    print("📊 Generating barchart...")
    fig_bar = topic_model.visualize_barchart(top_n_topics=30, n_words=10)
    bar_path = os.path.join(output_dir, "topic_barchart.html")
    fig_bar.write_html(bar_path)
    visuals.append(bar_path)

    print("🗺️ Generating intertopic distance map...")
    fig_map = topic_model.visualize_topics()
    map_path = os.path.join(output_dir, "topic_map.html")
    fig_map.write_html(map_path)
    visuals.append(map_path)

    print("🔥 Generating topic similarity heatmap...")
    fig_heatmap = topic_model.visualize_heatmap()
    heatmap_path = os.path.join(output_dir, "topic_heatmap.html")
    fig_heatmap.write_html(heatmap_path)
    visuals.append(heatmap_path)

    print("🌳 Generating hierarchical topics...")
    fig_hierarchy = topic_model.visualize_hierarchy()
    hierarchy_path = os.path.join(output_dir, "topic_hierarchy.html")
    fig_hierarchy.write_html(hierarchy_path)
    visuals.append(hierarchy_path)

    print(f"\n✅ All visualizations saved in: {output_dir}")

    # Open files in browser
    for path in visuals:
        print(f"🌐 Opening {os.path.basename(path)}...")
        webbrowser.open("file://" + os.path.abspath(path))

# generate_topic_visualizations(
#     model_path="../main_topic_models/bertopic_all-mpnet-base-v2_final_output/bertopic_final.model",
#     mapping_path="../main_topic_models/bertopic_all-mpnet-base-v2_final_output/topic_doc_map.pkl",
#     output_dir="../visualizations",
# )

for vis in os.listdir("../visualizations"):
    if vis.endswith(".html"):
        print(f"🌐 Opening {vis}...")
        webbrowser.open("file://" + os.path.abspath(os.path.join("../visualizations", vis)))

# Keyword Export

In [ ]:
import pandas as pd
import json
import csv
import ast
import os

# Load the topics CSV file
topics_df = pd.read_csv('topic_modelling_v2/round_2/topics.csv')

# Create a formatted export of topic keywords
output_lines = []
output_lines.append("# Topic Model Keywords Export")
output_lines.append("")
output_lines.append("| Topic ID | Count | Name | Top Keywords |")
output_lines.append("|---------|-------|------|-------------|")

for _, row in topics_df.iterrows():
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    # Convert the string representation of list to an actual list
    keywords = eval(row['Representation'])
    # Format keywords as a comma-separated string
    keywords_str = ', '.join(keywords)

    # Add line to output
    output_lines.append(f"| {topic_id} | {count} | {name} | {keywords_str} |")

# Write the output to a file
output_path = "topic_modelling_v2/round_2/topic_keywords_export.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write('\n'.join(output_lines))

print(f"Topic keywords export created at {output_path}")

# Set paths
input_file = 'topic_modelling_v2/round_2/topics.csv'
output_file = 'topic_modelling_v2/round_2/topic_keywords.csv'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Read the CSV file
df = pd.read_csv(input_file)

# Create a new DataFrame to store the keywords
keywords_df = pd.DataFrame()

# Extract topic numbers and names
keywords_df['Topic'] = df['Topic']
keywords_df['Name'] = df['Name']

# Parse the Representation column and extract keywords
keywords_df['Keywords'] = df['Representation'].apply(lambda x: ', '.join(ast.literal_eval(x)))

# Write to CSV
keywords_df.to_csv(output_file, index=False)

print(f"Keywords exported to {output_file}")

# Post-Length Distribution Check

In [ ]:
import json
import tiktoken
import pickle
import os
import sys

file_path = "topic_modelling_v2/round_4/preprocessed-docs.pkl"  # Path to your Reddit JSONL or Pickle file

# Parameters for comparison
NEWS_ARTICLE_COUNT = 10_000
AVG_NEWS_WORDS = 250  # typical words per news article

# Tokenizer for LLM token counts
enc = tiktoken.get_encoding("cl100k_base")  # OpenAI's GPT-3.5/4 tokenizer

total_words = 0
total_tokens = 0
item_count = 0

# Determine file type and process accordingly
if file_path.endswith('.jsonl'):
    # Process Reddit JSONL file
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                post = json.loads(line)
                text = ""

                # Adjust these keys depending on your Reddit data fields
                if "title" in post:
                    text += post["title"] + " "
                if "selftext" in post and post["selftext"]:
                    text += post["selftext"]

                words = text.strip().split()
                tokens = enc.encode(text)

                total_words += len(words)
                total_tokens += len(tokens)
                item_count += 1
            except json.JSONDecodeError:
                continue
    item_type = "post"

elif file_path.endswith('.pkl'):
    # Process pickle file
    try:
        with open(file_path, "rb") as f:
            data = pickle.load(f)

        # Process all documents, handling different possible structures
        all_docs = []

        # Case 1: data is a dict where values are lists of strings (documents)
        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, list):
                    all_docs.extend([doc for doc in value if isinstance(doc, str)])

        # Case 2: data is a list of strings (documents)
        elif isinstance(data, list):
            all_docs.extend([doc for doc in data if isinstance(doc, str)])

        # Process all documents
        for text in all_docs:
            words = text.strip().split()
            tokens = enc.encode(text)

            total_words += len(words)
            total_tokens += len(tokens)
            item_count += 1

    except Exception as e:
        print(f"Error reading or processing pickle file: {e}")

    item_type = "document"

else:
    print(f"Error: Unsupported file type. Please use .jsonl or .pkl files.")
    sys.exit(1)

if item_count == 0:
    print("No items found!")
else:
    avg_words = total_words / item_count
    avg_tokens = total_tokens / item_count

    news_words_total = NEWS_ARTICLE_COUNT * AVG_NEWS_WORDS
    news_tokens_total = news_words_total * 1.33  # approx token conversion

    print(f"📊 {item_type.capitalize()} corpus size:")
    print(f"- {item_type.capitalize()}s: {item_count}")
    print(f"- Total words: {total_words:,}")
    print(f"- Total tokens: {total_tokens:,}")
    print(f"- Avg words/{item_type}: {avg_words:.1f}")
    print(f"- Avg tokens/{item_type}: {avg_tokens:.1f}\n")

    print(f"📑 News articles equivalent (~{AVG_NEWS_WORDS} words each):")
    print(f"- News total words: {news_words_total:,}")
    print(f"- News total tokens: {int(news_tokens_total):,}\n")

    ratio_words = total_words / news_words_total
    ratio_tokens = total_tokens / news_tokens_total
    print(f"🔍 Word count ratio ({item_type.capitalize()}s / News): {ratio_words:.2f}x")
    print(f"🔍 Token count ratio ({item_type.capitalize()}s / News): {ratio_tokens:.2f}x")